# 📊 ANÁLISIS DE DATOS DE SALARIOS — WAGE11
## Preprocesamiento, Análisis Exploratorio y Modelado Predictivo

---

### 📋 Descripción del Proyecto

Este notebook realiza un **análisis completo, documentado y justificado** del dataset `Wage11.xlsx`, que contiene información salarial de **935 trabajadores** con **15 variables**.

| Variable | Descripción |
|----------|-------------|
| `wage` | Salario mensual (variable objetivo) |
| `educ` | Años de educación formal |
| `exper` | Años de experiencia laboral |
| `IQ` | Cociente intelectual |
| `KWW` | Puntaje de conocimiento del mundo laboral |
| `hours` | Horas semanales trabajadas |
| `tenure` | Años de antigüedad en empresa actual |
| `age` | Edad del trabajador |
| `married` | Estado civil (1=casado) |
| `black` | Raza (1=afroamericano) |
| `south` | Región (1=sur) |
| `urban` | Zona (1=urbano) |
| `sibs` | Número de hermanos |
| `CAT` | Categoría laboral |

**Etapas:** Carga → Exploración → Limpieza → Imputación → Normalización → Feature Engineering → EDA → Modelado → Evaluación → Conclusiones

## 📦 SECCIÓN 1: IMPORTAR LIBRERÍAS

Se importan todas las librerías necesarias para el análisis:
- **`pandas` / `numpy`**: manipulación y cómputo de datos
- **`matplotlib` / `seaborn`**: visualizaciones estadísticas
- **`sklearn`**: preprocesamiento y modelado de Machine Learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuración visual global
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.float_format', '{:.2f}'.format)

print('✓ Librerías importadas correctamente')

## 📂 SECCIÓN 2: CARGAR DATOS

Carga del archivo `Wage11.xlsx` y primera inspección estructural del dataset.

In [ ]:
file_path = 'Wage11.xlsx'
df = pd.read_excel(file_path)
df_original = df.copy()

print('=' * 60)
print('  ARCHIVO CARGADO EXITOSAMENTE')
print('=' * 60)
print(f'  Filas     : {df.shape[0]}')
print(f'  Columnas  : {df.shape[1]}')
print(f'  Variables : {list(df.columns)}')

In [ ]:
# Primeras filas del dataset
print('PRIMERAS 10 FILAS:')
df.head(10)

In [ ]:
# Información estructural: tipos de datos y valores no nulos
print('INFORMACIÓN DEL DATASET:')
print(df.info())

In [ ]:
# Estadísticas descriptivas completas
print('ESTADÍSTICAS DESCRIPTIVAS:')
df.describe(include='all')

## 📊 SECCIÓN 3: ANÁLISIS VISUAL INICIAL

Antes de limpiar los datos, exploramos visualmente la distribución de las variables clave y la composición del dataset por grupos.

In [ ]:
# Distribución de WAGE (variable objetivo)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['wage'].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['wage'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Media: {df["wage"].mean():.0f}')
axes[0].axvline(df['wage'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Mediana: {df["wage"].median():.0f}')
axes[0].set_title('Distribución del Salario (WAGE)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Salario ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

axes[1].boxplot(df['wage'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot de WAGE (detección de outliers)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Salario ($)')

plt.suptitle('Variable Objetivo: WAGE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución por estado civil (gráfico de torta)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Torta: casados vs no casados
married_counts = df['married'].value_counts()
married_labels = ['No Casado', 'Casado']
axes[0].pie(married_counts.values, labels=married_labels,
            autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[0].set_title('Estado Civil', fontweight='bold', fontsize=12)

# Torta: zona urbana vs rural
urban_counts = df['urban'].value_counts()
axes[1].pie(urban_counts.values, labels=['Urbano' if i==1 else 'Rural' for i in urban_counts.index],
            autopct='%1.1f%%', colors=['#99ff99', '#ffcc99'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Zona Geográfica', fontweight='bold', fontsize=12)

# Torta: norte vs sur
south_counts = df['south'].value_counts()
axes[2].pie(south_counts.values, labels=['Norte/Otros' if i==0 else 'Sur' for i in south_counts.index],
            autopct='%1.1f%%', colors=['#c2c2f0', '#ffb3e6'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[2].set_title('Región', fontweight='bold', fontsize=12)

plt.suptitle('Composición del Dataset por Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuciones de variables numéricas clave
vars_num = ['educ', 'exper', 'IQ', 'KWW', 'age', 'hours']
labels_num = ['Educación (años)', 'Experiencia (años)', 'Cociente Intelectual',
              'KWW Score', 'Edad', 'Horas Semanales']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, var, lbl in zip(axes.flat, vars_num, labels_num):
    ax.hist(df[var].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df[var].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Media: {df[var].mean():.1f}')
    ax.axvline(df[var].median(), color='orange', linestyle='--', linewidth=1.5,
               label=f'Mediana: {df[var].median():.1f}')
    ax.set_title(lbl, fontweight='bold')
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribuciones de Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Salario promedio por grupos
from matplotlib.ticker import FuncFormatter

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Por estado civil
married_wage = df.groupby('married')['wage'].mean()
bars0 = axes[0].bar(['No Casado', 'Casado'], married_wage.values,
                    color=['#ff9999', '#66b3ff'], edgecolor='white', alpha=0.9)
axes[0].set_title('Salario Promedio por Estado Civil', fontweight='bold')
axes[0].set_ylabel('Salario Promedio ($)')
for bar in bars0:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'${bar.get_height():.0f}', ha='center', fontweight='bold')

# Por zona
urban_wage = df.groupby('urban')['wage'].mean()
bars1 = axes[1].bar(['Rural', 'Urbano'], urban_wage.values,
                    color=['#ffcc99', '#99ff99'], edgecolor='white', alpha=0.9)
axes[1].set_title('Salario Promedio: Rural vs Urbano', fontweight='bold')
axes[1].set_ylabel('Salario Promedio ($)')
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'${bar.get_height():.0f}', ha='center', fontweight='bold')

# Por región
south_wage = df.groupby('south')['wage'].mean()
bars2 = axes[2].bar(['Norte/Otros', 'Sur'], south_wage.values,
                    color=['#c2c2f0', '#ffb3e6'], edgecolor='white', alpha=0.9)
axes[2].set_title('Salario Promedio: Norte vs Sur', fontweight='bold')
axes[2].set_ylabel('Salario Promedio ($)')
for bar in bars2:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'${bar.get_height():.0f}', ha='center', fontweight='bold')

plt.suptitle('Análisis de Salario por Grupos Demográficos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧹 SECCIÓN 4: DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS

**¿Por qué eliminar duplicados?**

Los registros duplicados generan múltiples problemas:
1. **Sesgo estadístico**: las estadísticas descriptivas (media, varianza) quedan infladas hacia los valores duplicados
2. **Data leakage en el modelo**: si el mismo registro aparece en train y test, el modelo parece rendir mejor de lo que realmente es
3. **Distorsión de correlaciones**: las correlaciones entre variables se ven afectadas artificialmente

**Estrategia:** Se excluye la columna `id` (identificador único por diseño) y se conserva el **primer** registro de cada grupo duplicado.

In [ ]:
# Detectar duplicados (excluyendo columna 'id')
cols_sin_id = df.columns.difference(['id'])
duplicados = df.duplicated(subset=cols_sin_id)
num_duplicados = duplicados.sum()

print('=' * 60)
print('  DETECCIÓN DE DUPLICADOS')
print('=' * 60)
print(f'  Duplicados encontrados : {num_duplicados}')
print(f'  Porcentaje             : {num_duplicados/len(df)*100:.2f}%')

if num_duplicados > 0:
    print(f'\n  Muestra de registros duplicados:')
    print(df[df.duplicated(subset=cols_sin_id, keep=False)].head(4).to_string())
else:
    print('\n  No se encontraron registros duplicados.')

In [ ]:
# Eliminar duplicados y resetear índice
df = df.drop_duplicates(subset=df.columns.difference(['id']), keep='first').reset_index(drop=True)

print('RESULTADO:')
print(f'  Registros originales          : {len(df_original)}')
print(f'  Registros después de limpieza : {len(df)}')
print(f'  Duplicados eliminados         : {len(df_original) - len(df)}')
print(f'  Dimensiones finales           : {df.shape}')

## 🔍 SECCIÓN 5: ANÁLISIS DE VALORES FALTANTES

Identificación de variables con datos incompletos (NaN). Conocer el porcentaje y patrón de los faltantes es clave para elegir la estrategia de imputación correcta.

In [ ]:
# Tabla completa de valores faltantes
faltantes = pd.DataFrame({
    'Variable'   : df.columns,
    'Tipo'       : df.dtypes.values,
    'Faltantes'  : df.isnull().sum().values,
    'Porcentaje' : (df.isnull().sum().values / len(df) * 100).round(2)
})
faltantes = faltantes.sort_values('Faltantes', ascending=False)

print('TABLA DE VALORES FALTANTES:')
print(faltantes.to_string(index=False))
print(f'\nTotal valores faltantes en el dataset: {df.isnull().sum().sum()}')

In [ ]:
# Visualización: heatmap de valores faltantes
faltantes_plot = faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=True)

if len(faltantes_plot) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Barras horizontales
    bars = axes[0].barh(faltantes_plot['Variable'], faltantes_plot['Porcentaje'],
                        color='salmon', edgecolor='white', alpha=0.85)
    axes[0].set_title('Porcentaje de Valores Faltantes', fontweight='bold')
    axes[0].set_xlabel('Porcentaje (%)')
    for bar, pct in zip(bars, faltantes_plot['Porcentaje']):
        axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{pct}%', va='center', fontsize=10)

    # Heatmap
    vars_faltantes = faltantes_plot['Variable'].tolist()
    muestra = df[vars_faltantes].iloc[:100].isnull().astype(int)
    sns.heatmap(muestra.T, cmap='YlOrRd', cbar=False, ax=axes[1],
                linewidths=0.1, linecolor='lightgray')
    axes[1].set_title('Heatmap de Faltantes (primeras 100 filas)', fontweight='bold')
    axes[1].set_xlabel('Observación')

    plt.tight_layout()
    plt.show()
else:
    print('No hay valores faltantes — dataset completo ✓')

## 🔧 SECCIÓN 6: IMPUTACIÓN DE VALORES FALTANTES

**Estrategia de imputación:**

| Variable | Método | Justificación |
|----------|--------|---------------|
| `wage`, `exper`, `sibs`, `black` | **Mediana** | Distribuciones asimétricas con outliers — la mediana es robusta |
| `educ` | **Mediana** | Variable entera con distribución discreta |
| `IQ`, `KWW` | **Media** | Distribuciones aproximadamente simétricas (tipo escala de puntaje) |

La mediana es **resistente a outliers** mientras que la media puede verse sesgada por valores extremos.

In [ ]:
# Imputación con MEDIANA
vars_mediana = ['wage', 'educ', 'exper', 'black', 'sibs']
print('Imputación con MEDIANA:')
for var in vars_mediana:
    n = df[var].isnull().sum()
    if n > 0:
        val = df[var].median()
        df[var] = df[var].fillna(val)
        print(f'  {var:<12} → mediana = {val:.2f}  ({n} valores imputados)')
    else:
        print(f'  {var:<12} → sin faltantes')

# Imputación con MEDIA
vars_media = ['IQ', 'KWW']
print('\nImputación con MEDIA:')
for var in vars_media:
    n = df[var].isnull().sum()
    if n > 0:
        val = df[var].mean()
        df[var] = df[var].fillna(val)
        print(f'  {var:<12} → media = {val:.2f}  ({n} valores imputados)')
    else:
        print(f'  {var:<12} → sin faltantes')

print(f'\n✓ Faltantes totales después de imputación: {df.isnull().sum().sum()}')

## ⚖️ SECCIÓN 7: NORMALIZACIÓN Y ESTANDARIZACIÓN

Las variables tienen escalas muy distintas (ej: wage en cientos vs married en 0/1). Esto puede perjudicar ciertos algoritmos. Aplicamos dos técnicas:

| Técnica | Fórmula | Variable | Justificación |
|---------|---------|----------|---------------|
| **MinMaxScaler** | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | `wage` | Escala a [0,1], útil para comparar magnitudes |
| **StandardScaler** | $x' = \frac{x - \mu}{\sigma}$ | `KWW` | Media=0, Std=1; elimina efecto de escala en el modelo |

In [ ]:
# MinMaxScaler para WAGE → wage_normalizado
scaler_wage = MinMaxScaler()
df['wage_normalizado'] = scaler_wage.fit_transform(df[['wage']])

# StandardScaler para KWW → KWW_estandarizado
scaler_kww = StandardScaler()
df['KWW_estandarizado'] = scaler_kww.fit_transform(df[['KWW']])

print('WAGE — Normalización MinMaxScaler [0, 1]:')
print(f'  Original   → min: {df["wage"].min():.2f}  max: {df["wage"].max():.2f}  media: {df["wage"].mean():.2f}')
print(f'  Norm [0,1] → min: {df["wage_normalizado"].min():.4f}  max: {df["wage_normalizado"].max():.4f}  media: {df["wage_normalizado"].mean():.4f}')

print('\nKWW — Estandarización StandardScaler (z-score):')
print(f'  Original  → media: {df["KWW"].mean():.2f}  std: {df["KWW"].std():.2f}')
print(f'  Estándar  → media: {df["KWW_estandarizado"].mean():.6f}  std: {df["KWW_estandarizado"].std():.4f}')

In [ ]:
# Visualización: Antes vs Después de transformaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].hist(df['wage'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('WAGE — Original', fontweight='bold'); axes[0,0].set_xlabel('Salario ($)')

axes[0,1].hist(df['wage_normalizado'], bins=40, color='seagreen', edgecolor='white', alpha=0.85)
axes[0,1].set_title('WAGE — Normalizado [0,1]', fontweight='bold'); axes[0,1].set_xlabel('Valor')

axes[1,0].hist(df['KWW'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[1,0].set_title('KWW — Original', fontweight='bold'); axes[1,0].set_xlabel('KWW')

axes[1,1].hist(df['KWW_estandarizado'], bins=30, color='seagreen', edgecolor='white', alpha=0.85)
axes[1,1].set_title('KWW — Estandarizado (z-score)', fontweight='bold'); axes[1,1].set_xlabel('Z-score')

plt.suptitle('Transformaciones: Antes vs Después', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## ⚙️ SECCIÓN 8: INGENIERÍA DE CARACTERÍSTICAS

Creación de nuevas variables derivadas que capturan relaciones no lineales importantes:

| Feature | Fórmula | Interpretación |
|---------|---------|----------------|
| `salary_per_hour` | `wage / hours` | Productividad salarial por hora trabajada |
| `educ_exper_ratio` | `educ / (exper + 1)` | Balance entre educación y experiencia (+1 evita división por cero) |

Estas features pueden capturar relaciones que las variables originales por separado no expresan.

In [ ]:
# Feature 1: Salario por hora
df['salary_per_hour'] = df['wage'] / df['hours'].replace(0, np.nan)

# Feature 2: Ratio educación / experiencia
df['educ_exper_ratio'] = df['educ'] / (df['exper'] + 1)

print('NUEVAS VARIABLES CREADAS:')
print(f'\n  salary_per_hour  → promedio: {df["salary_per_hour"].mean():.4f}  mediana: {df["salary_per_hour"].median():.4f}')
print(f'  educ_exper_ratio → promedio: {df["educ_exper_ratio"].mean():.4f}  mediana: {df["educ_exper_ratio"].median():.4f}')

# Ver primeras filas con nuevas columnas
print('\nMuestra con features nuevas:')
df[['wage', 'hours', 'salary_per_hour', 'educ', 'exper', 'educ_exper_ratio']].head(8)

In [ ]:
# Distribución de las nuevas features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['salary_per_hour'].dropna(), bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[0].axvline(df['salary_per_hour'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Media: {df["salary_per_hour"].mean():.2f}')
axes[0].set_title('Distribución: Salario por Hora', fontweight='bold')
axes[0].set_xlabel('salary_per_hour'); axes[0].set_ylabel('Frecuencia'); axes[0].legend()

axes[1].hist(df['educ_exper_ratio'].dropna(), bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].axvline(df['educ_exper_ratio'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Media: {df["educ_exper_ratio"].mean():.2f}')
axes[1].set_title('Distribución: Ratio Educación/Experiencia', fontweight='bold')
axes[1].set_xlabel('educ_exper_ratio'); axes[1].set_ylabel('Frecuencia'); axes[1].legend()

plt.suptitle('Features Creadas (Ingeniería de Características)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 🔬 SECCIÓN 9: ANÁLISIS EXPLORATORIO DE DATOS (EDA)

Análisis profundo de correlaciones y relaciones entre variables. Este paso es fundamental para **seleccionar las variables más influyentes** en el modelo.

In [ ]:
# Matriz de correlaciones completa (heatmap)
vars_corr = ['wage', 'educ', 'exper', 'IQ', 'KWW', 'age', 'hours',
             'tenure', 'married', 'black', 'south', 'urban',
             'salary_per_hour', 'educ_exper_ratio']

corr_matrix = df[vars_corr].corr()

plt.figure(figsize=(14, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matriz de Correlaciones entre Variables', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

In [ ]:
# Ranking de correlaciones con WAGE
vars_interes = ['educ', 'exper', 'IQ', 'KWW', 'age', 'hours', 'tenure',
                'married', 'black', 'south', 'urban', 'salary_per_hour', 'educ_exper_ratio']

correlaciones = df[vars_interes + ['wage']].corr()['wage'][vars_interes].sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['salmon' if c < 0 else 'steelblue' for c in correlaciones]
correlaciones.plot(kind='barh', ax=ax, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Variables con WAGE (Pearson)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente de Correlación')
for i, (val, lbl) in enumerate(zip(correlaciones, correlaciones.index)):
    ax.text(val + (0.005 if val >= 0 else -0.005), i, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout(); plt.show()

print('TOP 5 variables más correlacionadas con WAGE:')
print(correlaciones.abs().sort_values(ascending=False).head(5))

In [ ]:
# Scatter plots: WAGE vs variables más correlacionadas
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
vars_scatter = [
    ('educ', 'Educación (años)'),
    ('exper', 'Experiencia (años)'),
    ('IQ',   'Cociente Intelectual'),
    ('age',  'Edad'),
    ('KWW',  'KWW Score'),
    ('tenure','Antigüedad (años)')
]

for ax, (var, lbl) in zip(axes.flat, vars_scatter):
    validos = df[[var, 'wage']].dropna()
    ax.scatter(validos[var], validos['wage'], alpha=0.3, color='steelblue', s=15)
    m, b = np.polyfit(validos[var], validos['wage'], 1)
    x_r = np.linspace(validos[var].min(), validos[var].max(), 100)
    ax.plot(x_r, m*x_r + b, color='red', linewidth=2, label=f'β={m:.1f}')
    corr = validos.corr()['wage'][var]
    ax.set_title(f'WAGE vs {lbl}\n(r={corr:.3f})', fontweight='bold')
    ax.set_xlabel(lbl); ax.set_ylabel('Salario ($)')
    ax.legend(fontsize=9)

plt.suptitle('Relación entre WAGE y Variables Predictoras', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots de salario por nivel educativo
educ_groups, educ_labels = [], []
for e in sorted(df['educ'].dropna().unique()):
    vals = df[df['educ'] == e]['wage'].dropna()
    if len(vals) >= 5:
        educ_groups.append(vals.values)
        educ_labels.append(str(int(e)))

plt.figure(figsize=(14, 5))
bp = plt.boxplot(educ_groups, labels=educ_labels, patch_artist=True,
                 boxprops=dict(facecolor='steelblue', alpha=0.5),
                 medianprops=dict(color='red', linewidth=2))
plt.title('Distribución de Salarios por Años de Educación', fontsize=13, fontweight='bold')
plt.xlabel('Años de Educación')
plt.ylabel('Salario ($)')
plt.tight_layout(); plt.show()

## 📐 SECCIÓN 10: PREPARACIÓN Y DIVISIÓN DE DATOS

Se definen las variables predictoras (X) y la variable objetivo (y), y se dividen los datos en **entrenamiento (80%)** y **prueba (20%)**.

- **`random_state=42`**: garantiza reproducibilidad (los mismos datos siempre en train/test)
- **`test_size=0.2`**: el 20% de los datos se reserva para evaluar el modelo con datos nunca vistos

In [ ]:
# Variables predictoras
X = df[['educ', 'exper', 'hours', 'IQ', 'KWW', 'tenure', 'age',
        'married', 'black', 'south', 'urban',
        'salary_per_hour', 'educ_exper_ratio']].copy()
y = df['wage'].copy()

# Eliminar filas con NaN residuales
idx_validos = X.notna().all(axis=1) & y.notna()
X = X[idx_validos].reset_index(drop=True)
y = y[idx_validos].reset_index(drop=True)

# División 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('=' * 60)
print('  DIVISIÓN DE DATOS')
print('=' * 60)
print(f'  Total muestras          : {len(X)}')
print(f'  Variables predictoras   : {len(X.columns)}')
print(f'  Entrenamiento (80%)     : {len(X_train)} muestras')
print(f'  Prueba        (20%)     : {len(X_test)} muestras')
print(f'\n  Salario prom. train    : ${y_train.mean():.2f}')
print(f'  Salario prom. test     : ${y_test.mean():.2f}')

## 🤖 SECCIÓN 11: SELECCIÓN Y JUSTIFICACIÓN DEL MODELO

### ¿Por qué Regresión Lineal Múltiple?

Para predecir el **salario (variable continua)** en función de múltiples predictores, se eligió la **Regresión Lineal Múltiple** por las siguientes razones:

| Criterio | Justificación |
|----------|---------------|
| **Variable objetivo continua** | `wage` es numérica continua → regresión, no clasificación |
| **Relaciones aproximadamente lineales** | Los scatter plots muestran tendencias lineales (especialmente `educ`, `IQ`) |
| **Interpretabilidad** | Los coeficientes β indican el efecto directo de cada variable: *"un año más de educación aumenta el salario en X$"* |
| **Múltiples predictores** | Tenemos 13 variables predictoras que se incorporan simultáneamente |
| **Baseline sólido** | Es el modelo estándar para análisis salariales en economía laboral |

### Ecuación del modelo:

$$\hat{wage} = \beta_0 + \beta_1 \cdot educ + \beta_2 \cdot exper + \beta_3 \cdot IQ + \ldots + \beta_{13} \cdot educ\_exper\_ratio + \varepsilon$$

Donde cada $\beta_i$ representa el cambio esperado en el salario por una unidad de aumento en la variable $i$, manteniendo todas las demás constantes.

In [ ]:
# Crear y entrenar el modelo de Regresión Lineal
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print('✓ Modelo entrenado exitosamente')
print(f'\nIntercepto (β₀): {modelo.intercept_:.4f}')

# Tabla de coeficientes ordenada
coefs_df = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente (β)': modelo.coef_,
    'Efecto': ['↑ Sube salario' if c > 0 else '↓ Baja salario' for c in modelo.coef_]
}).sort_values('Coeficiente (β)', ascending=False)

print('\nCOEFICIENTES DEL MODELO:')
print(coefs_df.to_string(index=False))

In [ ]:
# Visualización de coeficientes
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if c > 0 else 'salmon' for c in coefs_df['Coeficiente (β)']]
ax.barh(coefs_df['Variable'], coefs_df['Coeficiente (β)'],
        color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes del Modelo — Impacto en el Salario', fontsize=13, fontweight='bold')
ax.set_xlabel('Valor del coeficiente β ($ por unidad de variable)')
for i, (val, lbl) in enumerate(zip(coefs_df['Coeficiente (β)'], coefs_df['Variable'])):
    ax.text(val + (5 if val >= 0 else -5), i, f'{val:.1f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout(); plt.show()

## 📈 SECCIÓN 12: EVALUACIÓN DEL MODELO

Se evalúa el modelo con métricas estándar:

| Métrica | Descripción | Fórmula |
|---------|-------------|--------|
| **R²** | % de varianza explicada por el modelo | $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$ |
| **MAE** | Error absoluto promedio en $ | $MAE = \frac{1}{n}\sum|y_i - \hat{y}_i|$ |
| **RMSE** | Error cuadrático medio (penaliza errores grandes) | $RMSE = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ |

Se comparan los resultados en **entrenamiento vs prueba** para detectar **overfitting** (cuando el modelo memoriza los datos de entrenamiento pero no generaliza bien).

In [ ]:
# Predicciones en train y test
y_train_pred = modelo.predict(X_train)
y_test_pred  = modelo.predict(X_test)

# Calcular métricas
r2_train   = r2_score(y_train, y_train_pred)
r2_test    = r2_score(y_test, y_test_pred)
mae_train  = mean_absolute_error(y_train, y_train_pred)
mae_test   = mean_absolute_error(y_test, y_test_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('=' * 65)
print(f'  {"MÉTRICA":<35} {"TRAIN":>12}  {"TEST":>10}')
print('=' * 65)
print(f'  {"R² (Coef. Determinación) [%]":<35} {r2_train*100:>11.2f}%  {r2_test*100:>9.2f}%')
print(f'  {"MAE (Error Absoluto Medio) [$]":<35} ${mae_train:>11.2f}  ${mae_test:>9.2f}')
print(f'  {"RMSE (Raíz Error Cuadrático) [$]":<35} ${rmse_train:>11.2f}  ${rmse_test:>9.2f}')
print('=' * 65)

diff = r2_train - r2_test
print(f'\n  Diferencia R² train-test (overfitting): {diff:.4f}')
print('  → ' + ('EXCELENTE: Sin overfitting significativo ✓' if diff < 0.05 else
                'ACEPTABLE: Overfitting mínimo' if diff < 0.10 else
                'ADVERTENCIA: Posible overfitting'))

In [ ]:
# Real vs Predicho
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (y_real, y_pred, titulo, color) in zip(axes, [
    (y_test, y_test_pred, f'TEST — R²={r2_test:.4f}', 'steelblue'),
    (y_train, y_train_pred, f'TRAIN — R²={r2_train:.4f}', 'seagreen')
]):
    ax.scatter(y_real, y_pred, alpha=0.35, color=color, s=20)
    mn, mx = min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Predicción perfecta')
    ax.set_title(f'Real vs Predicho ({titulo})', fontweight='bold')
    ax.set_xlabel('Valor Real ($)'); ax.set_ylabel('Valor Predicho ($)')
    ax.legend()

plt.suptitle('Comparación: Valores Reales vs Predichos', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Análisis de Residuos
residuos = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuos vs predichos
axes[0].scatter(y_test_pred, residuos, alpha=0.4, color='steelblue', s=20)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residuos vs Valores Predichos', fontweight='bold')
axes[0].set_xlabel('Valores Predichos ($)'); axes[0].set_ylabel('Residuos ($)')

# Distribución de residuos
axes[1].hist(residuos, bins=35, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(residuos.mean(), color='orange', linestyle='--', linewidth=2,
                label=f'Media={residuos.mean():.2f}')
axes[1].set_title('Distribución de Residuos', fontweight='bold')
axes[1].set_xlabel('Residuo ($)'); axes[1].set_ylabel('Frecuencia'); axes[1].legend()

plt.suptitle('Análisis de Residuos — Verificación de Supuestos', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Media de residuos : {residuos.mean():.4f}  (debe ser ≈ 0 para modelo insesgado)')
print(f'Std  de residuos  : {residuos.std():.4f}')

## 📝 SECCIÓN 13: CONCLUSIONES Y RECOMENDACIONES

Resumen ejecutivo de todo el proceso de análisis y principales hallazgos.

In [ ]:
print('=' * 70)
print('  CONCLUSIONES FINALES DEL ANÁLISIS WAGE11')
print('=' * 70)

print(f'''
  1. CALIDAD Y LIMPIEZA DE DATOS
     Registros originales          : {len(df_original)}
     Registros después de limpieza : {len(df)}
     Duplicados eliminados         : {len(df_original) - len(df)}
     Dataset final                 : {df.shape[0]} filas × {df.shape[1]} columnas

  2. TRANSFORMACIONES APLICADAS
     Normalización (MinMaxScaler)  : wage → wage_normalizado [0,1]
     Estandarización (z-score)     : KWW  → KWW_estandarizado
     Features creadas              : salary_per_hour, educ_exper_ratio

  3. HALLAZGOS DEL EDA
     Variables más correlac. con wage: educ, IQ, exper, tenure, age
     Trabajadores urbanos ganan más que rurales
     Mayor educación → mayor salario (relación positiva)

  4. MODELO: REGRESIÓN LINEAL MÚLTIPLE
     Variables predictoras         : {len(X.columns)}
     Muestras entrenamiento        : {len(X_train)}
     Muestras prueba               : {len(X_test)}

  5. RENDIMIENTO DEL MODELO
     R² Test (varianza explicada)  : {r2_test*100:.2f}%
     MAE Test (error promedio)     : ${mae_test:.2f}
     RMSE Test                     : ${rmse_test:.2f}
     Overfitting (R²train - R²test): {r2_train - r2_test:.4f}
''')

print('  TOP 5 VARIABLES MÁS INFLUYENTES:')
for i, (_, row) in enumerate(coefs_df.head(5).iterrows(), 1):
    print(f'     {i}. {row["Variable"]:<25} β = {row["Coeficiente (β)"]:+.2f}')

print()
print('  RECOMENDACIONES:')
print('  1. Probar modelos no lineales (Random Forest, XGBoost) para mejorar R²')
print('  2. Incluir más variables (sector industria, tipo contrato) para mejorar el modelo')
print('  3. Segmentar el análisis por región (norte/sur) ya que hay diferencias salariales')
print('=' * 70)

## 💾 SECCIÓN 14: GUARDAR ARCHIVO PROCESADO

Se guarda el dataset limpio, imputado y con las nuevas features en un archivo Excel.

In [ ]:
output_file = 'Wage11_PROCESADO.xlsx'
df.to_excel(output_file, index=False)

print('=' * 60)
print('  ARCHIVO PROCESADO GUARDADO')
print('=' * 60)
print(f'  Archivo   : {output_file}')
print(f'  Registros : {len(df)}')
print(f'  Columnas  : {len(df.columns)}')
print(f'\nColumnas en el archivo:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')
print('\n✓ Análisis completado exitosamente')